# Tracing Basics

### Setup

Make sure you set your environment variables, including your OpenAI API key.

In [ ]:
# You can set them inline
# import os
# os.environ["OPENAI_API_KEY"] = "your openai api key"
# os.environ["LANGSMITH_API_KEY"] = "your langsmith api key"
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"

In [2]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

### Tracing with @traceable

@traceable 装饰器是一种从 LangSmith Python SDK 记录跟踪信息的简单方法。只需用 @traceable 装饰任何函数即可。

装饰器的工作原理是：每次函数被调用时，它都会创建一个运行树，并将其插入到当前跟踪中。然后，函数的输入、名称和其他信息会被传输到 LangSmith。如果函数引发错误或返回响应，这些信息也会被添加到运行树中，并且更新会同步到 LangSmith，以便您可以检测和诊断错误源。所有这些操作都在后台线程中执行，以避免阻塞应用程序的运行。

In [6]:
# TODO: Import traceable
from langsmith import traceable
from openai import OpenAI
from typing import List
import nest_asyncio
import os
from utils import get_vector_db_retriever

MODEL_PROVIDER = "qwen"
MODEL_NAME = "qwen3-max"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
nest_asyncio.apply()
retriever = get_vector_db_retriever()

# TODO: Set up tracing for each function
@traceable
def retrieve_documents(question: str):
    return retriever.invoke(question)   # NOTE: This is a LangChain vector db retriever, so this .invoke() call will be traced automatically

@traceable
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)

@traceable
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

@traceable
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


@traceable 会为你处理 RunTree 生命周期！

In [4]:
question = "How can I trace with the @traceable decorator?"
ai_answer = langsmith_rag(question)
print(ai_answer)

To trace with the `@traceable` decorator, simply decorate any function with `@traceable` from the LangSmith SDK. Ensure the `LANGSMITH_TRACING` environment variable is set to `'true'` and your `LANGSMITH_API_KEY` is configured. If using file paths in `Attachment` arguments, also set `dangerously_allow_filesystem=True` in the decorator.


##### Let's take a look in LangSmith!

### Adding Metadata

LangSmith 支持随跟踪信息一起发送任意元数据。

元数据（Metadata）是一组键值对，可以附加到运行记录上。元数据可用于存储有关运行记录的附加信息，例如生成该运行记录的应用程序版本、运行记录的生产环境，或您希望与运行记录关联的任何其他信息。与标签类似，您可以使用元数据在 LangSmith 用户界面中筛选运行记录，也可以使用元数据将运行记录分组以进行分析。

In [7]:
from langsmith import traceable

@traceable(
    # TODO: Add Metadata
    metadata={"vectordb": "sklearn"}
)
def retrieve_documents(question: str):
    return retriever.invoke(question)

@traceable
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)

@traceable(
    # TODO: Add Metadata
    metadata={"model_name": MODEL_NAME, "model_provider": MODEL_PROVIDER}
)
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

@traceable
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


In [8]:
question = "How do I add Metadata to a Run with @traceable?"
ai_answer = langsmith_rag(question)
print(ai_answer)

You can add metadata to a run with `@traceable` by specifying the `metadata` parameter in the decorator, like `@ls.traceable(metadata={"my-key": "my-value"})`. Additionally, you can dynamically update metadata during execution by accessing the current run tree via `ls.get_current_run_tree()` and modifying its `metadata` attribute. You can also pass metadata at invocation time using the `langsmith_extra` parameter when calling the traced function.


你还可以在运行时添加元数据！

In [9]:
question = "How do I add metadata at runtime?"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"runtime_metadata": "foo"}})
print(ai_answer)

To add metadata at runtime, include a `run_metadata` object in your run data with any key-value pairs you want to store. This field is optional and can contain arbitrary structured data. Ensure it's part of the run payload when serializing and uploading the run.


##### Let's take a look in LangSmith!